# KMeans features

**Iteration 5** — representation learning, added after the iteration-4 round.

Customer-archetype cluster distances, learned unsupervised on **internal** features only (no `EXT_SOURCE`, no target). A global clustering with `k` picked by silhouette, plus targeted clusterings on bureau / previous-application / payment subsets. All prefixed `x_kmeans_`; saved to `kmeans.pkl`.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from src.data import load_master, save_features

X, _ = load_master(all_rows=True)
idx = X.index
drop = [c for c in X.columns if c.lower().startswith(("ext_", "x_kmeans_", "x_pca_", "x_ae_")) or "ext_source" in c.lower()]
internal = X.drop(columns=drop)
del X                                               # free the full master immediately
keep = internal.columns[(internal.std(numeric_only=True) > 0) & (internal.isna().mean() < 0.9)]
internal = internal[keep]

def prep(df):
    df = df.astype("float32")
    med = df.median().astype("float32")             # keep everything float32 - no float64 upcast
    mean = df.mean().astype("float32")
    std = df.std().astype("float32").replace(0, 1)
    df = df.fillna(med)
    return ((df - mean) / std).fillna(0).to_numpy("float32")

Z = prep(internal)
out = pd.DataFrame(index=idx)
internal.shape

(356255, 3224)

## Global clustering (silhouette-picked k)

In [2]:
rng = np.random.RandomState(0)
samp = rng.choice(len(Z), min(10000, len(Z)), replace=False)
scores = {}
for k in [4, 6, 8, 12, 16]:
    scores[k] = silhouette_score(Z[samp], KMeans(k, random_state=0, n_init=5).fit_predict(Z[samp]))
    print(f"k={k:2d}  silhouette {scores[k]:.3f}")
best_k = max(scores, key=scores.get)
print("chosen k:", best_k)
fit_samp = rng.choice(len(Z), min(80_000, len(Z)), replace=False)
km = KMeans(best_k, random_state=0, n_init=5).fit(Z[fit_samp])   # fit on a subsample -> light
for i, col in enumerate(km.transform(Z).T):
    out[f"x_kmeans_d{i}"] = col

k= 4  silhouette 0.093
k= 6  silhouette 0.091
k= 8  silhouette 0.015
k=12  silhouette 0.016
k=16  silhouette 0.007
chosen k: 4


## Subset clusterings

In [3]:
subsets = {
    "bureau": [c for c in internal.columns if c.startswith(("bureau", "bb_"))],
    "prev":   [c for c in internal.columns if c.startswith("prev")],
    "pay":    [c for c in internal.columns if c.startswith(("ins_", "pos_", "cc_"))],
}
for name, cols in subsets.items():
    if len(cols) < 5:
        continue
    Zi = prep(internal[cols])
    ss = rng.choice(len(Zi), min(80_000, len(Zi)), replace=False)
    km = KMeans(6, random_state=0, n_init=5).fit(Zi[ss])
    for i, col in enumerate(km.transform(Zi).T):
        out[f"x_kmeans_{name}_d{i}"] = col
    del Zi
out.shape

(356255, 22)

# Save

In [4]:
save_features(out, "kmeans"); out.shape

(356255, 22)